# Exploration of Survey Property Maps — DP2 (USDF)

This notebook accesses the **survey property maps** of Rubin LSST DP2 from the `dp2_prep` butler.

Two types of data products are available in the collection `LSSTCam/runs/DRP/v30_0_4/DM-54249`:

1. **`SurveyWidePropertyMapPlot`** — PNG/PDF images of the maps generated by the pipeline (sections 3–5).
2. **`consolidated_map`** — queryable `HealSparseMap` objects (sections 6–10, if available).

**To be adapted**: collection names and dataset type names must be adjusted
according to what is actually present in the butler at runtime.

| Dataset type | Description | Unit |
|---|---|---|
| `deepCoadd_exposure_time_consolidated_map_sum` | Total exposure time accumulated per sky position | **second** |
| `deepCoadd_epoch_consolidated_map_min` | Epoch of the earliest observation | **MJD** |
| `deepCoadd_epoch_consolidated_map_max` | Epoch of the latest observation | **MJD** |
| `deepCoadd_epoch_consolidated_map_mean` | Mean epoch of all contributing observations | **MJD** |
| `deepCoadd_psf_size_consolidated_map_weighted_mean` | Weighted mean of the PSF characteristic width (determinant radius) | **pixel** |
| `deepCoadd_psf_e1_consolidated_map_weighted_mean` | Weighted mean of PSF ellipticity component e1 | dimensionless |
| `deepCoadd_psf_e2_consolidated_map_weighted_mean` | Weighted mean of PSF ellipticity component e2 | dimensionless |
| `deepCoadd_psf_maglim_consolidated_map_weighted_mean` | Weighted mean of the PSF flux 5σ magnitude limit | **mag AB** |
| `deepCoadd_sky_background_consolidated_map_weighted_mean` | Weighted mean of the sky background light level | **nJy** |
| `deepCoadd_sky_noise_consolidated_map_weighted_mean` | Weighted mean of the standard deviation of the sky background level | **nJy** |
| `deepCoadd_dcr_dra_consolidated_map_weighted_mean` | Weighted mean of the DCR-induced astrometric shift in the RA direction, expressed as a proportionality factor | dimensionless |
| `deepCoadd_dcr_ddec_consolidated_map_weighted_mean` | Weighted mean of the DCR-induced astrometric shift in the Dec direction, expressed as a proportionality factor | dimensionless |
| `deepCoadd_dcr_e1_consolidated_map_weighted_mean` | Weighted mean of the DCR-induced change in PSF ellipticity (e1), expressed as a proportionality factor | dimensionless |
| `deepCoadd_dcr_e2_consolidated_map_weighted_mean` | Weighted mean of the DCR-induced change in PSF ellipticity (e2), expressed as a proportionality factor | dimensionless |

### Notes

- **DCR** = *Differential Chromatic Refraction* — an atmospheric effect that shifts and distorts the PSF
  as a function of source colour and zenith angle. The four DCR maps quantify the survey's sensitivity to this effect.
- **e1, e2** = PSF ellipticity components in Cartesian coordinates:
  e1 = (Ixx − Iyy) / (Ixx + Iyy),  e2 = 2·Ixy / (Ixx + Iyy).
- **`_weighted_mean`** = inverse-variance weighted mean (weight = 1/σ²) over all visits contributing to the coadd.
- **`_sum`** = direct sum over all contributing visits (used only for `exposure_time`).
- Physical format: FITS files with `.hsp` extension (HealSparse format), 84 files in DP1 (14 maps × 6 bands).
- All maps use dimensions `{band, skymap}` and are retrieved via `butler.get(map_name, dataId={'band': ..., 'skymap': ...})`.

- **Author:** Sylvie Dagoret-Campagne  
- **Creation date:** 2026-03-11  
- **Environment:** USDF RSP — `LSST` kernel (`lsst_distrib` stack)  
- **Reference DP1 notebook:** `203_1_Survey_property_maps.ipynb`  
- **Reference DP2 access notebook:** `2026-03-07_AccessToDP2.ipynb`
- **DP2 DRP Campaigns on Confluence:** https://rubinobs.atlassian.net/wiki/spaces/DM/pages/661192727/LSSTCam+Intermittent+DRP+Runs
- **DP2-DRP Plots:** https://usdf-rsp.slac.stanford.edu/plot-navigator

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.lines as mlines
from matplotlib.colors import LogNorm, SymLogNorm, Normalize
import pandas as pd
import os
from IPython.display import display
from pathlib import Path

# LSST middleware
from lsst.daf.butler import Butler

# Astropy — used to compute Galactic plane coordinates for overlay
from astropy.coordinates import SkyCoord, Galactic
import astropy.units as u

# skyproj: HEALPix / HealSparse map visualisation
# run: pip install skyproj  if not available in the environment
try:
    import skyproj
    HAS_SKYPROJ = True
    print("skyproj available")
except ImportError:
    HAS_SKYPROJ = False
    print("skyproj not available — all-sky visualisations will be disabled")

# healsparse (optional — useful to inspect HealSparseMap objects)
try:
    import healsparse
    HAS_HEALSPARSE = True
    print("healsparse available")
except ImportError:
    HAS_HEALSPARSE = False
    print("healsparse not available")

In [ ]:
# ── Output directories ────────────────────────────────────────────────────────
NB_TAG   = 'DP2_SPMAP_01'
DIR_DATA = f'data_{NB_TAG}'
DIR_FIGS = f'figs_{NB_TAG}'
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f'Data : {os.path.abspath(DIR_DATA)}')
print(f'Figs : {os.path.abspath(DIR_FIGS)}')

## 2. Butler configuration

The `dp2_prep` repository is used instead of `dp1`.  
**TODO**: adjust `COLLECTION` to the exact collection name  
that contains the survey property maps in DP2.

In [ ]:
# ── Butler repository at USDF ─────────────────────────────────────────────────
REPO = 'dp2_prep'

# ── Target collection for survey property maps ────────────────────────────────
COLLECTION = 'LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545'

# ── Skymap ────────────────────────────────────────────────────────────────────
SKYMAP_NAME = 'lsst_cells_v2'
INSTRUMENT  = 'LSSTCam'

# ── Photometric bands ─────────────────────────────────────────────────────────
BANDS    = ['u', 'g', 'r', 'i', 'z', 'y']
BAND_REF = 'r'

# ── Reference field coordinates ───────────────────────────────────────────────
RA_ECDFS  = 53.13
DEC_ECDFS = -28.10

RA_COSMOS  = 150.12
DEC_COSMOS = 2.21

FIELD   = "COSMOS"
RA_CEN  = RA_COSMOS
DEC_CEN = DEC_COSMOS

## 3. Utility function definitions

In [ ]:
def get_plot_path(dataset_type, band, skymap=SKYMAP_NAME, collection=COLLECTION):
    """
    Return the POSIX path of the PNG file for a SurveyWidePropertyMapPlot.
    """
    try:
        uri = butler.getURI(
            dataset_type,
            dataId={'band': band, 'skymap': skymap},
            collections=collection
        )
        return uri.path
    except Exception as e:
        print(f"  Not found ({band}): {type(e).__name__}: {e}")
        return None


def show_plot(dataset_type, band, skymap=SKYMAP_NAME, collection=COLLECTION, figsize=(14, 8)):
    """
    Display a SurveyWidePropertyMapPlot for a given band.
    """
    path = get_plot_path(dataset_type, band, skymap, collection)
    if path is None:
        return
    img = mpimg.imread(path)
    short = (
        dataset_type
        .replace('propertyMapSurvey_deepCoadd_', '')
        .replace('_SurveyWidePropertyMapPlot', '')
    )
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"{short} — band={band}", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("Utility functions defined.")

## 4. Initialisation

In [ ]:
butler   = Butler(REPO)
registry = butler.registry
print(f"Butler instantiated on repository: {REPO}")

In [ ]:
try:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME, collections=COLLECTION)
except Exception as inst:
    print(type(inst))
    print(inst.args)
    print(inst)

In [ ]:
skymap

## 5. Explore available collections in `dp2_prep`

In [ ]:
all_collections = sorted(registry.queryCollections())
print(f"Total number of collections: {len(all_collections)}")

In [ ]:
keywords = ['survey', 'prop', 'map', 'consolidated', 'spm', 'dp2-prep']
spm_candidates = [
    c for c in all_collections
    if any(kw.lower() in c.lower() for kw in keywords)
]
print(f"Candidate collections ({len(spm_candidates)}):")
for c in spm_candidates:
    print(" ", c)

In [ ]:
drp_collections = [
    c for c in all_collections
    if 'DRP' in c or 'drp' in c
]
print(f"DRP collections ({len(drp_collections)}):")
#for c in drp_collections:
#    print(" ", c)

## 6. Search for survey property map dataset types

In [ ]:
print("=== Dataset types matching *consolidated_map* (all collections) ===")
for dtype in sorted(registry.queryDatasetTypes(expression="*consolidated_map*")):
    print(" ", dtype.name)

In [ ]:
patterns = [
    '*consolidated_map*',
    '*survey_property*',
    '*property_map*',
    '*healsparse*',
    '*healSparse*',
]

found_types = set()
for pattern in patterns:
    try:
        for dtype in registry.queryDatasetTypes(expression=pattern):
            found_types.add(dtype.name)
    except Exception as e:
        print(f"Pattern '{pattern}': {e}")

print(f"\nCandidate dataset types found ({len(found_types)}):")
for name in sorted(found_types):
    print(" ", name)

## 7. Separate map objects from plot images

In [ ]:
type_spmap  = []
type_spplot = []
for ftype in found_types:
    if "PropertyMapPlot" in ftype:
        type_spplot.append(ftype)
    elif "deepCoadd_" in ftype and "_map_" in ftype:
        type_spmap.append(ftype)

In [ ]:
print(f"\nDataset types present in collection: {COLLECTION}")
available_spm = []
for name in sorted(found_types):
    try:
        has_data = registry.queryDatasets(
            name,
            collections=COLLECTION
        ).any(execute=False, exact=False)
        if has_data:
            available_spm.append(name)
            print(f"  ✓ {name}")
    except Exception as e:
        print(f"  ? {name} — {e}")

print(f"\n→ {len(available_spm)} SPM dataset types available in {COLLECTION}")

In [ ]:
print(f"\nHealSparseMap dataset types present in collection: {COLLECTION}")
available_spmap = []
for name in sorted(type_spmap):
    try:
        has_data = registry.queryDatasets(
            name,
            collections=COLLECTION
        ).any(execute=False, exact=False)
        if has_data:
            available_spmap.append(name)
            print(f"  ✓ {name}")
    except Exception as e:
        print(f"  ? {name} — {e}")

print(f"\n→ {len(available_spmap)} HealSparseMap types available in {COLLECTION}")

In [ ]:
print(f"\nPlot dataset types present in collection: {COLLECTION}")
available_spplot = []
for name in sorted(type_spplot):
    try:
        has_data = registry.queryDatasets(
            name,
            collections=COLLECTION
        ).any(execute=False, exact=False)
        if has_data:
            available_spplot.append(name)
            print(f"  ✓ {name}")
    except Exception as e:
        print(f"  ? {name} — {e}")

print(f"\n→ {len(available_spplot)} Plot types available in {COLLECTION}")

## 8. Inspect dimensions of found dataset types

In [ ]:
for name in sorted(available_spm):
    dtype = butler.get_dataset_type(name)
    print(f"{name} :::", dtype)
    print(f"    required dimensions: {dtype.dimensions.required}")
    print()

## 9. Retrieve a Healpix format Survey Property Map dataset

In [ ]:
if available_spmap:
    MAP_NAME = available_spmap[0]
else:
    MAP_NAME = 'deepCoadd_psf_maglim_consolidated_map_weighted_mean'

print(f"Selected map : {MAP_NAME}")
print(f"Band         : {BAND_REF}")
print(f"Collection   : {COLLECTION}")

In [ ]:
hspmap = butler.get(MAP_NAME,
        dataId={'band': BAND_REF, 'skymap': SKYMAP_NAME},
        collections=COLLECTION)

In [ ]:
hspmap

In [ ]:
try:
    hspmap = butler.get(
        MAP_NAME,
        dataId={'band': BAND_REF, 'skymap': SKYMAP_NAME},
        collections=COLLECTION
    )
    print(f"Map retrieved successfully: {type(hspmap)}")
    print(hspmap)
except Exception as e:
    print(f"Retrieval error (expected for Plot types): {type(e).__name__}: {e}")
    hspmap = None

In [ ]:
HPMAP_DATASET_TYPES = [
    'deepCoadd_dcr_ddec_consolidated_map_weighted_mean',
    'deepCoadd_dcr_dra_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e1_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e2_consolidated_map_weighted_mean',
    'deepCoadd_epoch_consolidated_map_max',
    'deepCoadd_epoch_consolidated_map_mean',
    'deepCoadd_epoch_consolidated_map_min',
    'deepCoadd_exposure_time_consolidated_map_sum',
    'deepCoadd_psf_e1_consolidated_map_weighted_mean',
    'deepCoadd_psf_e2_consolidated_map_weighted_mean',
    'deepCoadd_psf_maglim_consolidated_map_weighted_mean',
    'deepCoadd_psf_size_consolidated_map_weighted_mean',
    'deepCoadd_sky_background_consolidated_map_weighted_mean',
    'deepCoadd_sky_noise_consolidated_map_weighted_mean',
]

print(f"Checking HealSparseMap availability in collection: {COLLECTION}")
rows_hpmap = []
for map_name in HPMAP_DATASET_TYPES:
    short = (
        map_name
        .replace('deepCoadd_', '')
        .replace('_consolidated', '')
    )
    for band in BANDS:
        ok = False
        try:
            hspmap_test = butler.get(
                map_name,
                dataId={'band': band, 'skymap': SKYMAP_NAME},
                collections=COLLECTION
            )
            ok = True
        except Exception:
            ok = False
        rows_hpmap.append({
            'dataset_type': map_name,
            'map': short,
            'band': band,
            'ok': ok
        })

df_hpmap = pd.DataFrame(rows_hpmap)
pivot_hpmap = df_hpmap.pivot(index='map', columns='band', values='ok')
print("HealSparseMap availability (True = retrievable):")
print(pivot_hpmap.to_string())
print(f"\nTotal available: {df_hpmap['ok'].sum()} / {len(df_hpmap)}")

HPMAPS_AVAILABLE = df_hpmap['ok'].any()

In [ ]:
PLOT_DATASET_TYPES = [
    'propertyMapSurvey_deepCoadd_dcr_ddec_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_dra_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_e1_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_e2_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_max_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_min_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_exposure_time_consolidated_map_sum_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_e1_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_e2_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_maglim_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_size_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_sky_background_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_sky_noise_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
]

rows = []
for name in PLOT_DATASET_TYPES:
    short = (
        name
        .replace('propertyMapSurvey_deepCoadd_', '')
        .replace('_SurveyWidePropertyMapPlot', '')
    )
    for band in BANDS:
        path = get_plot_path(name, band)
        rows.append({
            'dataset_type': name,
            'map': short,
            'band': band,
            'path': path,
            'ok': path is not None
        })

df = pd.DataFrame(rows)
pivot = df.pivot(index='map', columns='band', values='ok')
print("Available plots (True = file found):")
print(pivot.to_string())
print(f"\nTotal available: {df['ok'].sum()} / {len(df)}")

PLOTS_AVAILABLE = df['ok'].any()

## 9A. Configuration: option flags for HealSparse map display

Two display modes are available depending on data availability:

- **`SHOW_HPMAP_GRID`**: Display a 2×3 grid of HealSparse maps (one per band) with the Galactic plane overlaid.
- **`SHOW_PLOTS_IF_NO_HPMAP`**: Fall back to PNG plots if hpmaps are absent.
- **`USE_LOG_NORM`**: Apply a logarithmic colour scale to HealSparse maps.  
  Useful when DDF (Deep Drilling Fields) dominate the dynamic range (e.g. `exposure_time`).  
  Maps with signed values (DCR, PSF ellipticities) automatically switch to `SymLogNorm`.
- **`LOG_LINTHRESH`**: Linear threshold for `SymLogNorm` (values smaller than this
  in absolute value are rendered on a linear scale to avoid log(0) issues).

In [ ]:
# ── Display option flags ─────────────────────────────────────────────────────

SHOW_HPMAP_GRID        = True
SHOW_PLOTS_IF_NO_HPMAP = True

MAP_TO_DISPLAY  = 'deepCoadd_psf_maglim_consolidated_map_weighted_mean'
PLOT_TO_DISPLAY = 'propertyMapSurvey_deepCoadd_psf_maglim_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot'

# ── Logarithmic colour scale ──────────────────────────────────────────────────
# Set USE_LOG_NORM=True to reveal WFD structure when DDFs dominate the range.
USE_LOG_NORM  = True

# Linear threshold for SymLogNorm: values |x| < LOG_LINTHRESH use a linear scale.
# Adapt to the typical noise level of the map (e.g. 1e-3 for PSF ellipticities,
# 1.0 for sky_background in nJy, 10.0 for exposure_time in seconds).
LOG_LINTHRESH = 1.0

# Maps whose values can be negative or zero require SymLogNorm instead of LogNorm.
# All other maps use LogNorm (strictly positive values assumed).
SYMLOG_MAPS = {
    'deepCoadd_dcr_ddec_consolidated_map_weighted_mean',
    'deepCoadd_dcr_dra_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e1_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e2_consolidated_map_weighted_mean',
    'deepCoadd_psf_e1_consolidated_map_weighted_mean',
    'deepCoadd_psf_e2_consolidated_map_weighted_mean',
}

print(f'SHOW_HPMAP_GRID        = {SHOW_HPMAP_GRID}')
print(f'SHOW_PLOTS_IF_NO_HPMAP = {SHOW_PLOTS_IF_NO_HPMAP}')
print(f'MAP_TO_DISPLAY         = {MAP_TO_DISPLAY}')
print(f'USE_LOG_NORM           = {USE_LOG_NORM}')
print(f'LOG_LINTHRESH          = {LOG_LINTHRESH}')
print(f'HAS_SKYPROJ            = {HAS_SKYPROJ}')
print(f'HPMAPS_AVAILABLE       = {HPMAPS_AVAILABLE}')
print(f'PLOTS_AVAILABLE        = {PLOTS_AVAILABLE}')

## 9B. Galactic plane helper function

In [ ]:
def galactic_plane_radec(n_points=1000):
    """
    Return (ra, dec) arrays tracing the Galactic equator (b = 0) in ICRS.
    """
    gal_lon = np.linspace(0, 360, n_points)
    gal_lat = np.zeros(n_points)
    coords = SkyCoord(
        l=gal_lon * u.deg,
        b=gal_lat * u.deg,
        frame=Galactic
    ).icrs
    return coords.ra.deg, coords.dec.deg


GP_RA, GP_DEC = galactic_plane_radec(n_points=2000)
print(f'Galactic plane: {len(GP_RA)} sample points computed.')

## 9C. Display HealSparse maps — 2×3 grid (6 bands) with Galactic plane

The colour scale is controlled by `USE_LOG_NORM` (§9A):
- `USE_LOG_NORM=False` → linear `Normalize` (default matplotlib behaviour)
- `USE_LOG_NORM=True`  → `LogNorm` for positive-definite maps (exposure_time, psf_size…)
  or `SymLogNorm` for signed maps (DCR, PSF ellipticities)

The normalisation range is derived per map from the 1st–99th percentile of
valid pixels in the **first available band**, then applied to all 6 bands for
a consistent colour scale across the grid.

In [ ]:
def build_norm(hsp, map_name, use_log=True, linthresh=1.0):
    """
    Build a matplotlib Normalize instance appropriate for a HealSparseMap.

    Parameters
    ----------
    hsp : HealSparseMap
        A retrieved HealSparse map (used to sample valid pixel values).
    map_name : str
        Dataset type name — used to decide between LogNorm and SymLogNorm.
    use_log : bool
        If False, return a plain linear Normalize.
    linthresh : float
        Linear threshold for SymLogNorm (ignored for LogNorm).

    Returns
    -------
    norm : matplotlib Normalize (LogNorm, SymLogNorm, or Normalize)
    """
    # Extract all valid (non-sentinel) pixel values
    try:
        valid = hsp.get_values_flat(use_sentinel=False).astype(float)
    except Exception:
        # Fallback: no get_values_flat available — skip normalisation
        return Normalize()

    valid = valid[np.isfinite(valid)]
    if valid.size == 0:
        return Normalize()

    p01  = float(np.percentile(valid, 1))
    p99  = float(np.percentile(valid, 99))

    if not use_log:
        return Normalize(vmin=p01, vmax=p99)

    is_symlog = map_name in SYMLOG_MAPS

    if is_symlog:
        # Symmetric log scale: centre on 0, equal range on both sides
        vabs = max(abs(p01), abs(p99))
        return SymLogNorm(linthresh=linthresh, linscale=1.0,
                          vmin=-vabs, vmax=vabs)
    else:
        # Strictly positive log scale
        vmin_log = max(p01, 1e-6)   # guard against zero/negative
        return LogNorm(vmin=vmin_log, vmax=p99)


def display_hpmap_6bands(map_name, bands=BANDS, collection=COLLECTION,
                         skymap=SKYMAP_NAME, cmap='viridis',
                         figsize=(20, 12), lon_0=0.0,
                         use_log=True, linthresh=1.0):
    """
    Display a HealSparse map for all 6 Rubin bands in a 2x3 subplot grid.

    The Galactic plane (b=0) is overlaid as a dashed white line.
    The colour normalisation (linear / log / symlog) is shared across all
    6 bands and is computed from the first successfully retrieved band.

    Parameters
    ----------
    map_name : str
        HealSparse dataset type name.
    bands : list of str
        List of 6 photometric bands.
    collection : str
        Butler collection.
    skymap : str
        Skymap name.
    cmap : str
        Matplotlib colormap.
    figsize : tuple
        Figure size in inches.
    lon_0 : float
        Central longitude for the McBryde projection.
    use_log : bool
        If True, use LogNorm or SymLogNorm depending on the map type.
        If False, use a plain linear scale.
    linthresh : float
        Linear threshold for SymLogNorm.
    """
    if not HAS_SKYPROJ:
        print(f'skyproj not available — skipping: {map_name}')
        return

    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes_flat = axes.flatten()

    short = (
        map_name
        .replace('deepCoadd_', '')
        .replace('_consolidated_map_', ' | ')
    )

    # ── Pre-compute Galactic plane segments ───────────────────────────────────
    sort_idx     = np.argsort(GP_RA)
    gp_ra_s      = GP_RA[sort_idx]
    gp_dec_s     = GP_DEC[sort_idx]
    gaps         = np.where(np.diff(gp_ra_s) > 180)[0] + 1
    segments_ra  = np.split(gp_ra_s,  gaps)
    segments_dec = np.split(gp_dec_s, gaps)

    first_sp = None
    norm     = None    # shared normalisation — built from the first retrieved band

    for idx, (band, ax) in enumerate(zip(bands, axes_flat)):
        try:
            hsp = butler.get(
                map_name,
                dataId={'band': band, 'skymap': skymap},
                collections=collection
            )


            
            hsp_filename = f"{DIR_DATA}/hsp_{band}_{map_name}_{skymap}.fits"
            hsp.write(hsp_filename , clobber=True)
            print(f"HSP map file saved in  : {hsp_filename}")

            # Build shared norm once from the first retrieved band
            if norm is None:
                norm = build_norm(hsp, map_name,
                                  use_log=use_log, linthresh=linthresh)
                #norm_label = type(norm).__name__
                #print(f'  Colour norm: {norm_label}  '
                #      f'(vmin={norm.vmin:.3g}, vmax={norm.vmax:.3g})')

                # Remplacez le bloc d'affichage par celui-ci :
            if norm is not None:
                norm_label = type(norm).__name__
                vmin_str = f"{norm.vmin:.3g}" if norm.vmin is not None else "None"
                vmax_str = f"{norm.vmax:.3g}" if norm.vmax is not None else "None"
                print(f'  Colour norm: {norm_label} (vmin={vmin_str}, vmax={vmax_str})')
            
            sp = skyproj.McBrydeSkyproj(ax=ax, lon_0=lon_0)
            sp.draw_hspmap(hsp, cmap=cmap, norm=norm)
            sp.draw_colorbar(label=map_name.split('_')[-2])

            # ── Galactic plane overlay ────────────────────────────────────
            for seg_ra, seg_dec in zip(segments_ra, segments_dec):
                sp.ax.plot(seg_ra, seg_dec,
                           color='white', linestyle='--', linewidth=1.2,
                           transform=sp.ax.get_transform('world'))

            # Annotate the colour scale type in the subplot title
            scale_tag = f'[{type(norm).__name__}]' if use_log else '[linear]'
            ax.set_title(f'band = {band}  {scale_tag}',
                         fontsize=11, fontweight='bold')

            if first_sp is None:
                first_sp = sp

        except Exception as e:
            ax.set_visible(False)
            print(f'  [{band}] Error: {type(e).__name__}: {e}')

    # ── Legend (proxy artist) ─────────────────────────────────────────────────
    if first_sp is not None:
        proxy = mlines.Line2D([], [], color='white', linestyle='--',
                              linewidth=1.2, label='Galactic plane (b=0)')
        first_sp.ax.legend(handles=[proxy], loc='lower left', fontsize=9,
                           framealpha=0.4)

    fig.suptitle(
        f'Survey Property Map — {short}\n{collection}',
        fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()


# ── Display the selected map for all 6 bands ──────────────────────────────────
if SHOW_HPMAP_GRID and HPMAPS_AVAILABLE:
    print(f'Displaying HealSparse map: {MAP_TO_DISPLAY}')
    display_hpmap_6bands(MAP_TO_DISPLAY, lon_0=0.0,
                         use_log=USE_LOG_NORM, linthresh=LOG_LINTHRESH)
else:
    if not HPMAPS_AVAILABLE:
        print(f'HealSparse maps not found in collection {COLLECTION}.')
        print('-> Set SHOW_PLOTS_IF_NO_HPMAP=True and run section 9D for PNG fallback.')
    else:
        print('SHOW_HPMAP_GRID is False — display skipped.')

### 9C-bis. Loop over all available HealSparse map types

In [ ]:
if SHOW_HPMAP_GRID and HPMAPS_AVAILABLE:
    available_hpmap_names = (
        df_hpmap[df_hpmap['ok']]['dataset_type'].unique().tolist()
    )
    print(f'Displaying {len(available_hpmap_names)} map types...')
    for map_name in available_hpmap_names:
        print(f'\n--- {map_name} ---')
        display_hpmap_6bands(map_name, lon_0=0.0,
                             use_log=USE_LOG_NORM, linthresh=LOG_LINTHRESH)
else:
    print('HealSparse grid display skipped (SHOW_HPMAP_GRID=False or no hpmaps).')

## 9D. Fallback: display PNG plots — 2×3 grid (6 bands)

Triggered only when `SHOW_PLOTS_IF_NO_HPMAP = True`
**and** `HPMAPS_AVAILABLE = False` **and** `PLOTS_AVAILABLE = True`.

In [ ]:
def display_plots_6bands(plot_name, df_plots, bands=BANDS, figsize=(20, 12)):
    """
    Display a SurveyWidePropertyMapPlot PNG image for all 6 bands
    in a 2x3 subplot grid.
    """
    df_sub = df_plots[(df_plots['dataset_type'] == plot_name) & df_plots['ok']]
    if df_sub.empty:
        print(f'No PNG images found for: {plot_name}')
        return

    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, layout='constrained')
    axes_flat = axes.flatten()

    short = (
        plot_name
        .replace('propertyMapSurvey_deepCoadd_', '')
        .replace('_SurveyWidePropertyMapPlot', '')
    )

    for idx, band in enumerate(bands):
        ax = axes_flat[idx]
        row = df_sub[df_sub['band'] == band]
        if row.empty:
            ax.set_visible(False)
            continue
        img = mpimg.imread(row.iloc[0]['path'])
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'band = {band}', fontsize=12, fontweight='bold')

    for idx in range(len(bands), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.suptitle(
        f'Survey Property Map Plots — {short}\n'
        f'(PNG images — no HealSparse maps available in this collection)',
        fontsize=13, fontweight='bold'
    )
    plt.show()


USE_PLOT_FALLBACK = (
    SHOW_PLOTS_IF_NO_HPMAP
    and (not HPMAPS_AVAILABLE)
    and PLOTS_AVAILABLE
)

if USE_PLOT_FALLBACK:
    print(f'Fallback mode: displaying PNG plots for {PLOT_TO_DISPLAY}')
    display_plots_6bands(PLOT_TO_DISPLAY, df)
elif SHOW_PLOTS_IF_NO_HPMAP and HPMAPS_AVAILABLE:
    print('HealSparse maps are available — no plot fallback needed.')
    print('Use section 9C to display the HealSparse maps.')
elif not PLOTS_AVAILABLE:
    print('Neither HealSparse maps nor PNG plots found in this collection.')
else:
    print('SHOW_PLOTS_IF_NO_HPMAP is False — fallback display skipped.')

In [ ]:
if USE_PLOT_FALLBACK:
    for plot_name in PLOT_DATASET_TYPES:
        df_check = df[(df['dataset_type'] == plot_name) & df['ok']]
        if df_check.empty:
            continue
        short = (
            plot_name
            .replace('propertyMapSurvey_deepCoadd_', '')
            .replace('_SurveyWidePropertyMapPlot', '')
        )
        print(f'\n--- {short} ---')
        display_plots_6bands(plot_name, df)
else:
    print('Fallback loop skipped.')

## 10. Display all bands for a selected map

In [ ]:
PLOT_NAME = 'propertyMapSurvey_deepCoadd_psf_maglim_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot'

df_map = df[(df['dataset_type'] == PLOT_NAME) & df['ok']]
n = len(df_map)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3 * nrows), layout="constrained")
axes = np.array(axes).flatten()

for idx, (_, row) in enumerate(df_map.iterrows()):
    img = mpimg.imread(row['path'])
    axes[idx].imshow(img)
    axes[idx].axis('off')
    axes[idx].set_title(f"band = {row['band']}", fontsize=13, fontweight='bold')

for idx in range(n, len(axes)):
    axes[idx].set_visible(False)

short = PLOT_NAME.replace('propertyMapSurvey_deepCoadd_', '').replace('_SurveyWidePropertyMapPlot', '')
plt.suptitle(short, fontsize=14, fontweight='bold', y=1.01)
plt.show()

## 11. Display all maps, all bands

In [ ]:
for map_name in df['map'].unique():
    df_sub = df[(df['map'] == map_name) & df['ok']]
    if df_sub.empty:
        continue

    n = len(df_sub)
    ncols = min(n, 3)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3 * nrows), layout="constrained")
    axes = np.array(axes).flatten()

    for idx, (_, row) in enumerate(df_sub.iterrows()):
        img = mpimg.imread(row['path'])
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(f"band = {row['band']}", fontsize=11, fontweight='bold')

    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle(map_name, fontsize=11, fontweight='bold')
    plt.show()

## 12. Gallery of all 14 maps — reference band

In [ ]:
df_r = df[(df['band'] == BAND_REF) & df['ok']]

ncols = 2
nrows = int(np.ceil(len(df_r) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows), layout="constrained")
axes = np.array(axes).flatten()

for idx, (_, row) in enumerate(df_r.iterrows()):
    img = mpimg.imread(row['path'])
    axes[idx].imshow(img)
    axes[idx].axis('off')
    axes[idx].set_title(row['map'], fontsize=8)

for idx in range(len(df_r), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle(
    f"Survey Property Map Plots — DP2 — band={BAND_REF}\n{COLLECTION}",
    fontsize=13, fontweight='bold', y=1.005
)
plt.show()

## 13. Search for raw HealSparseMap objects in other collections

In [ ]:
raw_map_names = [
    name
    .replace('propertyMapSurvey_', '')
    .replace('_SurveyWidePropertyMapPlot', '')
    for name in PLOT_DATASET_TYPES
]

all_collections = sorted(registry.queryCollections())
MAP_TO_SEARCH   = raw_map_names[7]

print(f"Searching for '{MAP_TO_SEARCH}'\nacross all collections in repository {REPO}:\n")
found_in = []
for coll in all_collections:
    try:
        has = registry.queryDatasets(
            MAP_TO_SEARCH, collections=coll
        ).any(execute=False, exact=False)
        if has:
            found_in.append(coll)
            print(f"  checkmark  {coll}")
    except Exception:
        pass

if not found_in:
    print("  No collection contains raw HealSparseMap objects.")
    print("  Only PNG plots are available in this repository.")

In [ ]:
hspmap = None
if found_in:
    coll_hsp = found_in[0]
    print(f"Retrieving from: {coll_hsp}")
    try:
        hspmap = butler.get(
            MAP_TO_SEARCH,
            dataId={'band': BAND_REF, 'skymap': SKYMAP_NAME},
            collections=coll_hsp
        )
        print(f"  type: {type(hspmap)}")
        print(hspmap)
        val = hspmap.get_values_pos(RA_CEN, DEC_CEN)
        print(f"  Map value at field {FIELD}: {val}")
    except Exception as e:
        print(f"  Error: {type(e).__name__}: {e}")
else:
    print("No HealSparseMap available — use the PNG plots (sections 9-12).")

## 14. Map value at the reference field position

In [ ]:
if hspmap is not None:
    val_center = hspmap.get_values_pos(RA_CEN, DEC_CEN)
    print(f"Map '{MAP_NAME}' ({BAND_REF}-band) at the centre of {FIELD}")
    print(f"  RA={RA_CEN}, Dec={DEC_CEN}  ->  {val_center}")
    val_north = hspmap.get_values_pos(180.0, +60.0)
    print(f"  Sentinel value (outside coverage, RA=180, Dec=+60) -> {val_north}")
else:
    print("Map not available — section skipped.")

## 15. Statistics over a region around the reference field

In [ ]:
if hspmap is not None:
    span_dec = 2.0
    span_ra  = span_dec / np.cos(np.deg2rad(DEC_CEN))

    ra  = np.linspace(RA_CEN  - span_ra,  RA_CEN  + span_ra,  250)
    dec = np.linspace(DEC_CEN - span_dec, DEC_CEN + span_dec, 250)
    x, y = np.meshgrid(ra, dec)

    values = hspmap.get_values_pos(x, y)
    sentinel = hspmap.get_values_pos(180.0, +60.0)
    values[values == sentinel] = np.nan

    print(f"Region {FIELD} — {MAP_NAME} ({BAND_REF}-band)")
    print(f"  min    = {np.nanmin(values):.4f}")
    print(f"  max    = {np.nanmax(values):.4f}")
    print(f"  mean   = {np.nanmean(values):.4f}")
    print(f"  median = {np.nanmedian(values):.4f}")
    print(f"  std    = {np.nanstd(values):.4f}")
else:
    print("Map not available — section skipped.")

## 16. Local visualisation (pcolormesh)

In [ ]:
if hspmap is not None:
    fig, ax = plt.subplots(figsize=(7, 6))
    pcm = ax.pcolormesh(x, y, values, cmap='viridis')
    fig.colorbar(pcm, ax=ax, label=f"{MAP_NAME.split('_')[-2]} ({BAND_REF}-band)")
    ax.set_xlabel("Right Ascension (deg)")
    ax.set_ylabel("Declination (deg)")
    ax.set_title(f"{MAP_NAME}\n{BAND_REF}-band — {FIELD}", fontsize=11)
    ax.invert_xaxis()
    plt.tight_layout()
    plt.show()
else:
    print(f"Map not available for field {FIELD} — visualisation skipped.")

## 17. All-sky visualisation (skyproj)

In [ ]:
if hspmap is not None and HAS_SKYPROJ:
    fig, ax = plt.subplots(figsize=(10, 5))
    sp = skyproj.McBrydeSkyproj(ax=ax, lon_0=RA_CEN)
    sp.draw_hspmap(hspmap)
    sp.draw_colorbar(label=f"{MAP_NAME} ({BAND_REF}-band)")
    ax.set_title(f"All-sky — {MAP_NAME} ({BAND_REF}-band)", fontsize=12)
    plt.show()
elif hspmap is not None:
    print("skyproj not available — all-sky visualisation skipped.")
else:
    print("Map not available — visualisation skipped.")

## 18. Loop over all available HealSparseMap objects

In [ ]:
if not available_spm:
    print("No SPM dataset types found — section skipped.")
else:
    n_maps = len(available_spm)
    print(f"Attempting to retrieve {n_maps} maps in band '{BAND_REF}'...\n")

    maps = {}
    for name in available_spm:
        try:
            m = butler.get(name, dataId={'band': BAND_REF, 'skymap': SKYMAP_NAME}, collections=coll_hsp)
            maps[name] = m
            print(f"  checkmark {name}")
        except Exception as e:
            print(f"  cross {name}  — {type(e).__name__}: {e}")

    print(f"\n{len(maps)}/{n_maps} maps retrieved successfully.")

In [ ]:
if maps:
    n_ok  = len(maps)
    ncols = 2
    nrows = int(np.ceil(n_ok / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4.5 * nrows))
    axes = np.array(axes).flatten()

    span_dec = 0.75
    span_ra  = span_dec / np.cos(np.deg2rad(DEC_CEN))
    ra_arr   = np.linspace(RA_CEN  - span_ra,  RA_CEN  + span_ra,  200)
    dec_arr  = np.linspace(DEC_CEN - span_dec, DEC_CEN + span_dec, 200)
    xg, yg   = np.meshgrid(ra_arr, dec_arr)

    for idx, (name, hsp) in enumerate(maps.items()):
        ax   = axes[idx]
        vals = hsp.get_values_pos(xg, yg).astype(float)
        sentinel = hsp.get_values_pos(180.0, +60.0)
        vals[vals == sentinel] = np.nan

        pcm = ax.pcolormesh(xg, yg, vals, cmap='viridis')
        fig.colorbar(pcm, ax=ax)
        short_name = name.replace('deepCoadd_', '').replace('_consolidated_map_', ' ')
        ax.set_title(f"{short_name}\n({BAND_REF}-band)", fontsize=9)
        ax.set_xlabel("RA (deg)")
        ax.set_ylabel("Dec (deg)")
        ax.invert_xaxis()

    for idx in range(n_ok, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle(f"Survey Property Maps DP2 — field {FIELD} — band {BAND_REF}",
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

## 19. Retrieve maps without specifying band (band-independent maps)

In [ ]:
if not available_spm:
    print("No SPM dataset types found — section skipped.")
else:
    print("Attempting retrieval without 'band' parameter:")
    for name in available_spm:
        try:
            m = butler.get(name, collections=COLLECTION)
            print(f"  checkmark {name}  (band-independent)")
        except TypeError as e:
            print(f"  — {name}  requires 'band': {e}")
        except Exception as e:
            print(f"  cross {name}: {type(e).__name__}: {e}")

## 20. Full dump of dataset types in the target collection

In [ ]:
EXCLUDE_SUFFIXES = (
    '_config', '_log', '_metadata', '_resource_usage',
    'Plot', 'Metric', 'metric'
)

print(f"Dataset types present in '{COLLECTION}' (excluding bookkeeping):")
science_dtypes = []
for dt in registry.queryDatasetTypes():
    if any(s in dt.name for s in EXCLUDE_SUFFIXES):
        continue
    try:
        if registry.queryDatasets(
            dt, collections=COLLECTION
        ).any(execute=False, exact=False):
            science_dtypes.append(dt.name)
    except Exception:
        pass

science_dtypes.sort()
for name in science_dtypes:
    print(" ", name)

print(f"\nTotal: {len(science_dtypes)} dataset types")

## 21. Notes and next steps

- Update `COLLECTION` with the correct DP2 collection after inspecting the output of **section 5**.
- If dataset type names differ from DP1 (e.g. `deepCoadd_*` -> `step3_*` or similar), adapt the search patterns in **section 6**.
- Once the correct maps are identified, this notebook can be extended with:
  - Comparison of DP1 vs DP2 for the same fields.
  - Analysis of spatial coverage per band.
  - Histograms of map value distributions.